In [59]:
!pip install -U langchain_community chromadb langchain langchain_openai openai tiktoken rank_bm25 sentence_transformers cohere langchain_cohere

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.2/42.2 kB 2.5 MB/s eta 0:00:00


In [2]:
documents = [
    "This is a list which containing sample documents.",
    "Keywords are important for keyword-based search.",
    "Document analysis involves extracting keywords.",
    "Keyword-based search relies on sparse embeddings.",
    "Understanding document structure aids in keyword extraction.",
    "Efficient keyword extraction enhances search accuracy.",
    "Semantic similarity improves document retrieval performance.",
    "Machine learning algorithms can optimize keyword extraction methods."
]

In [64]:
from langchain_openai import OpenAIEmbeddings
from langchain_openai import OpenAI
import os
from google.colab import userdata
from langchain_text_splitters import RecursiveCharacterTextSplitter, CharacterTextSplitter
from langchain_community.document_loaders import TextLoader
from langchain.docstore.document import Document
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder
from langchain_cohere import CohereRerank
import cohere

In [60]:
OPENAI_API_TOKEN=userdata.get('OPENAI_API_KEY')
COHERE_API_KEY = userdata.get('COHERE_API_KEY')
os.environ["OPENAI_API_KEY"] = OPENAI_API_TOKEN
os.environ["COHERE_API_KEY"] = COHERE_API_KEY

In [ ]:
loder=TextLoader('state_of_the_union.txt', encoding="utf8")
document=loder.load()
text_spliiter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_spliiter.split_documents(document)

In [8]:
embeddings = OpenAIEmbeddings()

In [13]:
loder=TextLoader(documents, encoding="utf8")
document = [Document(page_content=doc) for doc in documents]
text_spliiter = CharacterTextSplitter(chunk_size=500, chunk_overlap=50)
docs = text_spliiter.split_documents(document)

In [18]:
def create_embeddings(documents):
  vector = embeddings.embed_documents(documents)
  return vector
document_embedding = create_embeddings(documents)

In [19]:
document_embedding

[[-0.01485621277242899,
  0.010421626269817352,
  0.0011739123146981,
  -0.007929346524178982,
  -0.013651843182742596,
  0.013359452597796917,
  -0.0055032032541930676,
  0.0059243845753371716,
  -0.02517341449856758,
  -0.008312239311635494,
  0.0192838367074728,
  0.024073468521237373,
  -0.006676245480775833,
  0.00862551387399435,
  0.006742381490767002,
  0.019436994567513466,
  0.01516252662986517,
  -0.013777153566479683,
  0.0005799944628961384,
  -0.025716423988342285,
  -0.013818923383951187,
  -0.0009076282731257379,
  -0.026008814573287964,
  0.0078109982423484325,
  0.006714534480124712,
  -0.0029935198836028576,
  0.00568420672789216,
  -0.019506610929965973,
  -0.018615515902638435,
  -0.0024644327349960804,
  -0.001105165807530284,
  -0.01512075588107109,
  -0.012899981811642647,
  -0.02464432641863823,
  -0.01602577418088913,
  0.003982077818363905,
  4.503332820604555e-05,
  -0.012148121371865273,
  -0.005889576394110918,
  -0.018420590087771416,
  0.0151068326085805

In [20]:
len(document_embedding)

8

In [21]:
len(document_embedding[0]) # dimensions

1536

In [23]:
query = "Natural language processing techniques enhance keyword extraction efficiency."
query_embedding = embeddings.embed_query(query)

In [24]:
len(query_embedding)

1536

In [26]:
similarities = cosine_similarity(np.array([query_embedding]), document_embedding)

In [27]:
similarities

array([[0.76209476, 0.87142736, 0.88538559, 0.84574862, 0.89197516,
        0.93404064, 0.86264345, 0.93704575]])

In [29]:
most_similar_index = np.argmax(similarities)
most_similar_document = documents[most_similar_index]
most_similar_document

'Machine learning algorithms can optimize keyword extraction methods.'

In [31]:
similarity_score = similarities[0][most_similar_index]
similarity_score

np.float64(0.9370457465431745)

In [32]:
sorted_indices = np.argsort(similarities[0])[::-1]
ranked_documents = [(documents[i], similarities[0][i]) for i in sorted_indices]
ranked_documents

[('Machine learning algorithms can optimize keyword extraction methods.',
  np.float64(0.9370457465431745)),
 ('Efficient keyword extraction enhances search accuracy.',
  np.float64(0.9340406418882201)),
 ('Understanding document structure aids in keyword extraction.',
  np.float64(0.891975159988005)),
 ('Document analysis involves extracting keywords.',
  np.float64(0.8853855917372521)),
 ('Keywords are important for keyword-based search.',
  np.float64(0.8714273639241588)),
 ('Semantic similarity improves document retrieval performance.',
  np.float64(0.8626434453954237)),
 ('Keyword-based search relies on sparse embeddings.',
  np.float64(0.8457486187587462)),
 ('This is a list which containing sample documents.',
  np.float64(0.7620947612974038))]

In [33]:
print("Top 4 Documents:")
for rank, (document, similarity) in enumerate(ranked_documents[:4], start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

Top 4 Documents:
Rank 1: Document - 'Machine learning algorithms can optimize keyword extraction methods.', Similarity Score - 0.9370457465431745
Rank 2: Document - 'Efficient keyword extraction enhances search accuracy.', Similarity Score - 0.9340406418882201
Rank 3: Document - 'Understanding document structure aids in keyword extraction.', Similarity Score - 0.891975159988005
Rank 4: Document - 'Document analysis involves extracting keywords.', Similarity Score - 0.8853855917372521


In [37]:
top_4_documents = [doc[0] for doc in ranked_documents[:4]]
top_4_documents

['Machine learning algorithms can optimize keyword extraction methods.',
 'Efficient keyword extraction enhances search accuracy.',
 'Understanding document structure aids in keyword extraction.',
 'Document analysis involves extracting keywords.']

# Reranking using BM25

In [38]:
tokenized_top_4_documents = [doc.split() for doc in top_4_documents]
tokenized_top_4_documents

[['Machine',
  'learning',
  'algorithms',
  'can',
  'optimize',
  'keyword',
  'extraction',
  'methods.'],
 ['Efficient', 'keyword', 'extraction', 'enhances', 'search', 'accuracy.'],
 ['Understanding',
  'document',
  'structure',
  'aids',
  'in',
  'keyword',
  'extraction.'],
 ['Document', 'analysis', 'involves', 'extracting', 'keywords.']]

In [40]:
tokenized_query = query.split()
tokenized_query

['Natural',
 'language',
 'processing',
 'techniques',
 'enhance',
 'keyword',
 'extraction',
 'efficiency.']

In [41]:
bm25=BM25Okapi(tokenized_top_4_documents)
bm25

In [42]:
bm25_scores = bm25.get_scores(tokenized_query)
bm25_scores

array([0.16686672, 0.1907998 , 0.17803252, 0.        ])

In [44]:
sorted_indices_reranked = np.argsort(bm25_scores)[::-1]
sorted_indices_reranked

array([1, 2, 0, 3])

In [45]:
reranked_documents = [(top_4_documents[i], bm25_scores[i]) for i in sorted_indices_reranked]
reranked_documents

[('Efficient keyword extraction enhances search accuracy.',
  np.float64(0.19079979534096053)),
 ('Understanding document structure aids in keyword extraction.',
  np.float64(0.1780325227902643)),
 ('Machine learning algorithms can optimize keyword extraction methods.',
  np.float64(0.1668667199671815)),
 ('Document analysis involves extracting keywords.', np.float64(0.0))]

In [46]:
print("Rerank of top 4 Documents:")
for rank, (document, similarity) in enumerate(reranked_documents, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

Rerank of top 4 Documents:
Rank 1: Document - 'Efficient keyword extraction enhances search accuracy.', Similarity Score - 0.19079979534096053
Rank 2: Document - 'Understanding document structure aids in keyword extraction.', Similarity Score - 0.1780325227902643
Rank 3: Document - 'Machine learning algorithms can optimize keyword extraction methods.', Similarity Score - 0.1668667199671815
Rank 4: Document - 'Document analysis involves extracting keywords.', Similarity Score - 0.0


# Reranking using CrossEncoder

In [51]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [53]:
pairs = []
for doc in top_4_documents:
    pairs.append([query, doc])
pairs

[['Natural language processing techniques enhance keyword extraction efficiency.',
  'Machine learning algorithms can optimize keyword extraction methods.'],
 ['Natural language processing techniques enhance keyword extraction efficiency.',
  'Efficient keyword extraction enhances search accuracy.'],
 ['Natural language processing techniques enhance keyword extraction efficiency.',
  'Understanding document structure aids in keyword extraction.'],
 ['Natural language processing techniques enhance keyword extraction efficiency.',
  'Document analysis involves extracting keywords.']]

In [54]:
scores = cross_encoder.predict(pairs)
scores

array([ 0.842167 ,  3.1378722, -2.919299 , -2.8781896], dtype=float32)

In [55]:
scored_docs = zip(scores, top_4_documents)
reranked_document_cross_encoder = sorted(scored_docs, reverse=True)
reranked_document_cross_encoder

[(np.float32(3.1378722),
  'Efficient keyword extraction enhances search accuracy.'),
 (np.float32(0.842167),
  'Machine learning algorithms can optimize keyword extraction methods.'),
 (np.float32(-2.8781896), 'Document analysis involves extracting keywords.'),
 (np.float32(-2.919299),
  'Understanding document structure aids in keyword extraction.')]

In [56]:
print("Rerank of top 4 Documents using Cross Encoder:")
for rank, (document, similarity) in enumerate(reranked_document_cross_encoder, start=1):
    print(f"Rank {rank}: Document - '{document}', Similarity Score - {similarity}")

Rerank of top 4 Documents using Cross Encoder:
Rank 1: Document - '3.1378722190856934', Similarity Score - Efficient keyword extraction enhances search accuracy.
Rank 2: Document - '0.8421670198440552', Similarity Score - Machine learning algorithms can optimize keyword extraction methods.
Rank 3: Document - '-2.8781895637512207', Similarity Score - Document analysis involves extracting keywords.
Rank 4: Document - '-2.9192988872528076', Similarity Score - Understanding document structure aids in keyword extraction.


# Reranking using CohereAPI

In [65]:
co = cohere.Client()
response = co.rerank(
    model="rerank-english-v3.0",
    query="Natural language processing techniques enhance keyword extraction efficiency.",
    documents=top_4_documents,
    return_documents=True
)

In [66]:
print(response)

id='6171d0f5-93c6-40c1-9127-7b92894b5ec3' results=[RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Efficient keyword extraction enhances search accuracy.'), index=1, relevance_score=0.99411184), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Machine learning algorithms can optimize keyword extraction methods.'), index=0, relevance_score=0.9129032), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Understanding document structure aids in keyword extraction.'), index=2, relevance_score=0.32885265), RerankResponseResultsItem(document=RerankResponseResultsItemDocument(text='Document analysis involves extracting keywords.'), index=3, relevance_score=0.02865267)] meta=ApiMeta(api_version=ApiMetaApiVersion(version='1', is_deprecated=None, is_experimental=None), billed_units=ApiMetaBilledUnits(images=None, input_tokens=None, output_tokens=None, search_units=1.0, classifications=None), tokens=None, warnings=None)


In [67]:
for i in range(4):
  print(f'text: {response.results[i].document.text} score: {response.results[i].relevance_score}')

text: Efficient keyword extraction enhances search accuracy. score: 0.99411184
text: Machine learning algorithms can optimize keyword extraction methods. score: 0.9129032
text: Understanding document structure aids in keyword extraction. score: 0.32885265
text: Document analysis involves extracting keywords. score: 0.02865267
